In [ ]:
# Task 4: General Health Query Chatbot (Prompt Engineering Based)

In [ ]:
! pip install streamlit -q
! pip install transformers -q

In [ ]:
# 1 Install packages
!pip install streamlit transformers torch

# 2 Download cloudflared
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!mv cloudflared /usr/local/bin/

--2026-04-25 08:37:44--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-25 08:37:44--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-25T09%3A27%3A43Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-25T0

In [ ]:
%%writefile .env
OPENROUTER_API_KEY=""

Overwriting .env


In [ ]:
import os
from dotenv import load_dotenv

# Load the .env file
load_dotenv()

# Get API key
api_key = os.getenv("OPENROUTER_API_KEY")

# Test output
if api_key:
    print("✅ API key loaded successfully!")
    # Optional: print last 4 characters to check
    print("API key ends with:", api_key[-4:])
else:
    print("❌ API key not found. Check your .env file.")



✅ API key loaded successfully!
API key ends with: e967


In [ ]:
#===================================================================================
# we add 3 safety layers one can check wheather the query is health related or not
# second will be system prompt
# third will be filter on harmful words like dignose , cure, mg,tablets
#===================================================================================

%%writefile app.py

# =========================
# IMPORTS
# =========================
import streamlit as st
import requests
import os
from dotenv import load_dotenv

# =========================
# LOAD API KEY
# =========================
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")

URL = "https://openrouter.ai/api/v1/chat/completions"

# ==============================
# 🧠 STRICT HEALTH CLASSIFIER
# =============================

def is_health_related(query):

    prompt = f"""
You are a strict medical domain classifier.

Return ONLY ONE WORD:
YES or NO

RULES:

YES = HUMAN HEALTH ONLY
- symptoms (headache, fever, migraine)
- diseases (covid, dengue, flu)
- mental health (anxiety, stress, depression)
- body conditions (pain, infection, weakness)
- prevention and recovery

NO = EVERYTHING ELSE
- coding, python, AI, technology
- cooking, recipes
- education, english, math
- news, economy, politics
- entertainment, social media (TikTok etc.)

IMPORTANT RULE:
If question is about HUMAN BODY or MIND → ALWAYS YES

Question: {query}
"""

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "openai/gpt-3.5-turbo",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0
    }

    try:
        res = requests.post(URL, headers=headers, json=data)

        if res.status_code == 200:
            ans = res.json()["choices"][0]["message"]["content"]

            #  CLEAN RESPONSE
            ans = ans.strip().upper()

            #  STRICT CHECK (FIXED BUG)
            if ans == "YES":
                return True
            elif ans == "NO":
                return False
            else:
                #  unexpected output → safe fallback
                return True

    except Exception as e:
        print("Classifier error:", e)

    # 🛡 SAFE DEFAULT
    return True

# ==============================================
# 🏥 SYSTEM PROMPT (NO MEDICINE / NO DIAGNOSIS)
# ==============================================
SYSTEM_PROMPT = """
You are a STRICT HUMAN HEALTH ASSISTANT.

RULES:
- Only answer human health questions
- NEVER diagnose diseases
- NEVER prescribe medicines
- NEVER suggest drugs or dosages

ALLOWED:
- symptoms explanation
- general health information
- prevention tips
- lifestyle advice

If asked for cure or medicine:
→ give only general care (rest, hydration, sleep)
→ always suggest doctor
"""


# =========================
# 🤖 GPT RESPONSE
# =========================
def get_response(messages):

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "openai/gpt-3.5-turbo",
        "messages": messages
    }

    try:
        res = requests.post(URL, headers=headers, json=data)

        if res.status_code == 200:
            return res.json()["choices"][0]["message"]["content"]

    except:
        pass

    return "Error generating response"


# =========================
# 🌐 STREAMLIT UI
# =========================
st.set_page_config(page_title="Strict Health AI", page_icon="🏥")

st.title("🏥 HEALTH ASSISTANT ")
st.warning("Only human health questions allowed. Everything else is blocked.")

# =========================
# MEMORY
# =========================
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

# =========================
# CHAT HISTORY
# =========================
for msg in st.session_state.messages:
    if msg["role"] != "system":
        with st.chat_message(msg["role"]):
            st.write(msg["content"])

# =========================
# USER INPUT
# =========================
user_input = st.chat_input("Ask a health question...")

if user_input:

    with st.chat_message("user"):
        st.write(user_input)

    # =========================
    # STEP 1: HEALTH FILTER
    # =========================
    if not is_health_related(user_input):

        reply = "⚠️ BLOCKED: Only human health-related questions are allowed."

    else:

        # save user message
        st.session_state.messages.append({"role": "user", "content": user_input})

        with st.spinner("Thinking..."):
            reply = get_response(st.session_state.messages)

        # safety cleanup (NO MEDICINE / NO DIAGNOSIS)
        bad_words = ["prescribe", "medicine", "tablet", "mg", "drug", "diagnose"]

        if any(w in reply.lower() for w in bad_words):
            reply = """
⚠️ I cannot provide medicines or diagnosis.

General care:
- rest
- hydration
- sleep
- healthy diet

Please consult a doctor for proper guidance.
"""

        st.session_state.messages.append({"role": "assistant", "content": reply})

    with st.chat_message("assistant"):
        st.write(reply)




Writing app.py


In [ ]:
import subprocess
import time

# Start Streamlit without blocking
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"])

# Wait a few seconds for Streamlit to start
time.sleep(15)

In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-04-25T08:34:50Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-25T08:34:50Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-25T08:34:52Z INF +--------------------------------------------------------------------------------------------+
2026-04-25T08:34:52Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-25T08:34:52Z INF |  https://quote-nevertheless-favourite-wider.trycloudfl